## Notebook for visualizing anchor, buffer, and register tokens

In [ ]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from transformers import CLIPProcessor, CLIPModel

RETAIN = 192 # Original 576
OBJECT_LAYER = 9
ALPHA = 0.5

In [ ]:
image_path = "../assets/surf.webp"
image = Image.open(image_path)
image

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "openai/clip-vit-large-patch14-336"
model = CLIPModel.from_pretrained(model_name, output_attentions=True).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

inputs = processor(images=image, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.vision_model(**inputs, output_attentions=True)

attentions = outputs.attentions

### Anchor tokens

In [ ]:
mid_attn = attentions[OBJECT_LAYER]
mid_avg_attn = mid_attn.mean(dim=1).squeeze(0)  # (seq_len, seq_len)
attn_score = mid_avg_attn.sum(0)[1:]

anchor_indices = torch.topk(attn_score, k=int(RETAIN * ALPHA / 5)).indices.cpu()
anchor_indices

### Buffer tokens

In [ ]:
buffer_indices = torch.cat([anchor_indices-1,
                            anchor_indices+1,
                            anchor_indices-24,
                            anchor_indices+24])
valid_buffer = (buffer_indices >= 0) & (buffer_indices <= 575)
buffer_indices = buffer_indices[valid_buffer].unique().cpu()
buffer_indices = buffer_indices[~torch.isin(buffer_indices, anchor_indices)]
buffer_indices

### Register tokens

In [ ]:
chosens = torch.cat([anchor_indices, buffer_indices]).unique()
register_num = RETAIN - len(chosens)

deep_attn = attentions[-2] # LLaVA uses tokens from the last but one layer of CLIP
deep_attn = deep_attn.mean(1).squeeze(0)
deep_attn = deep_attn.sum(0)[1:]
deep_attn[chosens] = -1

register_indices = torch.topk(deep_attn, k=register_num).indices.cpu()
register_indices

In [ ]:
def visualize_token_categories(image: Image.Image,
                               anchor_indices: torch.Tensor,
                               buffer_indices: torch.Tensor,
                               register_indices: torch.Tensor,
                               grid_size: int = 24,
                               patch_alpha: float = 0.75):
    img_np = np.array(image)
    H, W = img_np.shape[:2]
    patch_h, patch_w = H // grid_size, W // grid_size

    fig, ax = plt.subplots()
    ax.imshow(img_np)
    ax.axis('off')

    def draw_boxes(indices, color):
        for idx in indices.tolist():
            row = idx // grid_size
            col = idx % grid_size
            rect = plt.Rectangle((col * patch_w, row * patch_h),
                                 patch_w, patch_h,
                                 linewidth=0,
                                 edgecolor=None,
                                 facecolor=color,
                                 alpha=patch_alpha)
            ax.add_patch(rect)

    draw_boxes(anchor_indices, color='#f3a361')
    draw_boxes(buffer_indices, color='#e66d50')
    draw_boxes(register_indices, color='#299d8f')

    plt.show()

visualize_token_categories(image, anchor_indices, buffer_indices, register_indices)